In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

In [10]:
CSV_PATH = "datasets/text_emotion.csv"  # download from the given link and save with this name
TEXT_COLUMN = "content"
LABEL_COLUMN = "sentiment"
BATCH_SIZE = 256
EPOCHS = 10

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Columns:", df.columns.tolist())
print(df.head())
# Drop rows with missing text or labels
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

Columns: ['tweet_id', 'sentiment', 'author', 'content']
     tweet_id   sentiment       author  \
0  1956967341       empty   xoshayzers   
1  1956967666     sadness    wannamama   
2  1956967696     sadness    coolfunky   
3  1956967789  enthusiasm  czareaquino   
4  1956968416     neutral    xkilljoyx   

                                             content  
0  @tiffanylue i know  i was listenin to bad habi...  
1  Layin n bed with a headache  ughhhh...waitin o...  
2                Funeral ceremony...gloomy friday...  
3               wants to hang out with friends SOON!  
4  @dannycastillo We want to trade with someone w...  


In [4]:
label_counts = df[LABEL_COLUMN].value_counts()
top_5_labels = label_counts.index[:5].tolist()
print("\nTop 5 labels:", top_5_labels)
df = df[df[LABEL_COLUMN].isin(top_5_labels)].reset_index(drop=True)
print("Data size after filtering to top 5 labels:", len(df))
texts = df[TEXT_COLUMN].astype(str).values
labels = df[LABEL_COLUMN].values


Top 5 labels: ['neutral', 'worry', 'happiness', 'sadness', 'love']
Data size after filtering to top 5 labels: 31313


In [5]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
# Vocabulary size: number of unique tokens + 1 (because word_index starts at 1)
vocab_size = len(tokenizer.word_index) + 1
print("\nVocabulary size:", vocab_size)
# Max sequence length
max_seq_len = max(len(seq) for seq in sequences)
print("Maximum sequence length:", max_seq_len)
# Pad sequences to the same length
X = pad_sequences(sequences, maxlen=max_seq_len, padding="post", truncating="post")


Vocabulary size: 40774
Maximum sequence length: 37


In [ ]:
label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(labels)  # integers 0..4
num_classes = len(label_encoder.classes_)
print("Number of classes:", num_classes, " (should be 5)")
y = tf.keras.utils.to_categorical(y_int, num_classes=num_classes)

Number of classes: 5  (should be 5)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y_int)
print("\nTrain size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


Train size: 21919
Test size: 9394


In [8]:
embedding_output_dim = 10
model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_output_dim, input_length=max_seq_len),
    # First LSTM with 128 units
    layers.LSTM(128, return_sequences=True),
    # Second LSTM with 64 units
    layers.LSTM(64),
    # Fully connected + dropout
    layers.Dense(100, activation="relu"),
    layers.Dropout(0.5),
    # Output layer: 5 units, softmax
    layers.Dense(num_classes, activation="softmax")
])
model.summary()

c:\Users\mosta\anaconda3\envs\tf_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [11]:
history = model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS, validation_split=0.1,verbose=1)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 13s 125ms/step - accuracy: 0.2908 - loss: 1.5649 - val_accuracy: 0.3294 - val_loss: 1.5512
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.3572 - loss: 1.4922 - val_accuracy: 0.3709 - val_loss: 1.4738
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.4375 - loss: 1.3520 - val_accuracy: 0.3449 - val_loss: 1.4896
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.5128 - loss: 1.1870 - val_accuracy: 0.3691 - val_loss: 1.5569
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 110ms/step - accuracy: 0.5816 - loss: 1.0565 - val_accuracy: 0.3180 - val_loss: 1.6940
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 111ms/step - accuracy: 0.6413 - loss: 0.9407 - val_accuracy: 0.2933 - val_loss: 1.7504
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 109ms/step - accuracy: 0.6752 - loss: 0.8674 - val_accuracy: 0.3276 - val_loss: 1.8759
Epoch 8/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 110ms/step - accuracy: 0.7245 - loss: 0.7679 - val_accuracy: 0

In [12]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f}")


Test accuracy: 0.3493
